# Baseline 00: No RAG (parametric memory only)

This notebook is a required deliverable (SPEC.md §7) but is **not** one of "the 10 patterns" --
it does not follow the mandatory 8-section pattern template from §8. It exists to answer one
question: does retrieval actually help on this corpus, or could the model just answer from what
it already knows?

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** (see `.env.example`).
Outputs shown below are from a mocked run, proving the code path works end to end -- they are
not real quality numbers. A real run needs `OPENAI_API_KEY` set.


## Reproducibility header (SPEC.md §11)

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: 0a53e25169d7b633246d89b6ae127e8fb0ed03fd


## What this baseline does

The LLM answers each question directly, with **no retrieved context at all** -- just the bare
question, using `prompts/generation_prompt.txt` (held constant, R4) with an empty `{context}`.
`retrieved_chunk_ids` is always `[]`, and `hit@k`/`mrr` are therefore always 0 for every question
-- that's expected, not a bug, since there's nothing to retrieve.


In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

import time
from pathlib import Path

from recipes import AnswerWithCitations
from recipes.llm import get_llm
from evals.run import load_corpus_by_id, load_qa_set, run_pattern

GENERATION_MODEL = "gpt-4.1-mini-2025-04-14"
prompt_template = Path("../prompts/generation_prompt.txt").read_text(encoding="utf-8")


def make_no_rag_pattern(llm):
    def retrieve_and_answer(question: str, k: int = 5) -> AnswerWithCitations:
        start = time.perf_counter()
        prompt = prompt_template.format(context="", question=question)
        response = llm.complete(prompt=prompt, model=GENERATION_MODEL, temperature=0.0)
        latency_ms = (time.perf_counter() - start) * 1000
        return AnswerWithCitations(
            answer=response.text,
            retrieved_chunk_ids=[],
            latency_ms=latency_ms,
            input_tokens=response.input_tokens,
            output_tokens=response.output_tokens,
            cached_input_tokens=response.cached_input_tokens,
        )
    return retrieve_and_answer


## Run on our eval set

In [3]:
import os
from recipes.llm import MockLLM

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")

llm = get_llm()  # generation backend
# Judging needs its own backend under mock: MockLLM's canned generation
# text isn't valid JSON, and the judge prompts require JSON output. See
# evals/judges.py's JudgeParseError and evals/run.py's per-question error
# isolation -- without this, every question would (correctly, but noisily)
# show up as "retrieval succeeded but judging failed."
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

pattern_fn = make_no_rag_pattern(llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="00_baseline_no_rag",
    judges_enabled=True,
)


=== 00_baseline_no_rag (n=18) ===
  hit@3: 0.000  [95% CI 0.000, 0.000]
  hit@10: 0.000  [95% CI 0.000, 0.000]
  mrr: 0.000  [95% CI 0.000, 0.000]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 0.0
  p95_latency_ms: 0.0
  usd_per_query: $0.00050
  eval_usd: $0.0090


## Discussion

With zero retrieval, `hit@k` and `mrr` are trivially 0 for every question by construction -- this
baseline isn't meant to compete on those metrics. Its purpose is the `faithfulness` and
`answer_relevance` columns: if this baseline's answers are already faithful and relevant (because
the model happens to know the answer from training data, or because it correctly says "I don't
have enough information"), that's a signal RAG isn't adding as much value on that particular
question as it might look like from hit@k alone. On this pilot corpus of unpublished-until-2025
arXiv papers the model has never seen in training, we'd expect this baseline to score poorly on
faithfulness under a **real** run (nothing to be faithful to except its own guesses) -- the mocked
numbers above don't demonstrate that; a real run does.
